# 📊 Conference Room Quality — Exploratory Data Analysis

This notebook explores:
1. Label distribution and statistics
2. Feature distributions and outliers
3. Feature–label correlation heatmap
4. Representative detection examples

> Run `python src/detect.py` and `python src/features.py` before this notebook.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

# Project root
ROOT = Path('.').resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
else:
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'src'))

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#0F0F1A',
    'axes.facecolor':   '#1A1A2E',
    'axes.edgecolor':   '#2D2D4E',
    'axes.labelcolor':  '#E8E8F0',
    'xtick.color':      '#E8E8F0',
    'ytick.color':      '#E8E8F0',
    'text.color':       '#E8E8F0',
    'grid.color':       '#2D2D4E',
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
})

PALETTE = ['#6C63FF', '#43D3A6', '#FFC857', '#FF6584', '#00C9FF']

print('Paths configured. ROOT =', ROOT)

## 1. Load Data

In [ ]:
labels = pd.read_csv(ROOT / 'data' / 'labels.csv')
features = pd.read_csv(ROOT / 'features' / 'features.csv')

merged = features.merge(labels, on='filename', how='inner')
score_cols = ['cleanliness_score', 'chair_alignment_score', 'clutter_score', 'overall_score']
feature_cols = [c for c in features.columns if c != 'filename']

print(f'Labels: {len(labels)} rows | Features: {len(features)} rows | Merged: {len(merged)} rows')
print(f'Feature columns: {feature_cols}')
merged.head()

## 2. Label Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, col, color in zip(axes, score_cols, PALETTE):
    ax.hist(merged[col], bins=15, color=color, edgecolor='none', alpha=0.85)
    mean_val = merged[col].mean()
    ax.axvline(mean_val, color='white', linestyle='--', linewidth=1.5, alpha=0.8,
               label=f'Mean={mean_val:.2f}')
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Score (1–5)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Score Distributions — All Dimensions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'eda_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Distributions

In [ ]:
n_feats = len(feature_cols)
ncols = 4
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
axes = axes.flatten()

for i, (ax, col) in enumerate(zip(axes, feature_cols)):
    color = PALETTE[i % len(PALETTE)]
    ax.hist(merged[col].fillna(0), bins=20, color=color, edgecolor='none', alpha=0.85)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xlabel('Value', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide extra axes
for ax in axes[len(feature_cols):]:
    ax.set_visible(False)

fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Heatmap — Features vs Scores

In [ ]:
corr_df = merged[feature_cols + score_cols].corr()
feat_score_corr = corr_df.loc[feature_cols, score_cols]

fig, ax = plt.subplots(figsize=(10, max(6, len(feature_cols) * 0.55)))

cmap = sns.diverging_palette(260, 20, as_cmap=True)
sns.heatmap(
    feat_score_corr,
    ax=ax,
    cmap=cmap,
    vmin=-1, vmax=1,
    annot=True, fmt='.2f',
    annot_kws={'size': 9},
    linewidths=0.5,
    linecolor='#0F0F1A',
    cbar_kws={'shrink': 0.8},
)

ax.set_title('Feature–Score Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Score Dimension', fontsize=10)
ax.set_ylabel('Feature', fontsize=10)
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop correlates with overall_score:')
print(feat_score_corr['overall_score'].abs().sort_values(ascending=False).to_string())

## 5. Scatter — Clutter vs Overall Score

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pairs = [
    ('clutter_object_count', 'overall_score', '#FF6584'),
    ('chair_spacing_variance', 'chair_alignment_score', '#6C63FF'),
    ('table_surface_occupancy', 'cleanliness_score', '#43D3A6'),
]

for ax, (xf, yf, color) in zip(axes, pairs):
    ax.scatter(merged[xf], merged[yf], alpha=0.7, color=color, s=60, edgecolors='none')
    # Trend line
    z = np.polyfit(merged[xf].fillna(0), merged[yf], 1)
    p = np.poly1d(z)
    x_line = np.linspace(merged[xf].min(), merged[xf].max(), 100)
    ax.plot(x_line, p(x_line), '--', color='white', alpha=0.6, lw=2, label='Trend')
    corr = merged[xf].corr(merged[yf])
    ax.set_title(f'{xf}\nvs {yf}\n(r={corr:.2f})', fontsize=10, fontweight='bold')
    ax.set_xlabel(xf, fontsize=9)
    ax.set_ylabel(yf, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Key Feature–Score Relationships', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'eda_scatter_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Detection Object Statistics

In [ ]:
det_path = ROOT / 'outputs' / 'detections' / 'detections.json'
if det_path.exists():
    with open(det_path) as f:
        all_dets = json.load(f)

    from collections import Counter
    class_counts = Counter()
    for dets in all_dets.values():
        for d in dets:
            class_counts[d['class']] += 1

    classes, counts = zip(*class_counts.most_common())

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(classes)))
    bars = ax.barh(classes, counts, color=colors, edgecolor='none')
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(cnt), va='center', fontsize=9, color='#E8E8F0')
    ax.set_xlabel('Total Detections Across All Images', fontsize=10)
    ax.set_title('Object Class Frequencies (YOLOv8 Detections)', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'eda_class_frequencies.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No detections.json found. Run python src/detect.py first.')

## 7. Label Statistics Summary

In [ ]:
print('=== Label Summary ===')
print(merged[score_cols].describe().round(3).to_string())
print('\n=== Feature Summary ===')
print(merged[feature_cols].describe().round(4).to_string())